# Particle–mesh intersections and local curvatures

This notebook is a toy example in which we aim to analyse whether the inner mitochondrial membrane (IMM) directly beneath an ATP synthase differs in curvature from the surrounding membrane area. To do so, we:

1. Find where particles intersect the membrane (hit triangles or vertices).
2. Grow disk-shaped regions on the mesh around each intersection, sized to the approximate in-plane extent of an ATP synthase.
3. Compare mean and Gaussian curvature in two regions: an inner disk that should encompass the ATP synthase particle (≤ ~9 nm) vs an outer annulus (~9–18 nm) right outside of it.

These radii were chosen based on an approximate radius of the ATP synthase base (~9 nm from subtomogram averaging, measured in ChimeraX) and a surrounding band at twice that radius.

### Data sources

All example data come from the public Chlamy dataset resources linked below.

| Dataset | What it is |
|---|---|
| **IMM mesh** | Screened-Poisson surface of the IMM with precomputed curvatures (see **compute_curvatures** tutorial for more details) |
| **ATP synthase motive list** | Particle positions and orientations from the Chlamy annotations repository, used as rays (particles → mesh) or an oriented point cloud (mesh → particles) |

**IMM.** MemBrain v2 segmentation [Lamm et al., 2025](https://doi.org/10.1101/2024.01.05.574336) on a cryoCARE-denoised *C. reinhardtii* tomogram ([EMPIAR-11830](https://www.ebi.ac.uk/empiar/EMPIAR-11830)); screened Poisson mesh [Kazhdan & Hoppe, 2013](https://doi.org/10.1109/TVCG.2012.34) (external MeshLab / PyMeshLab workflow); curvatures via [Rusinkiewicz, 2004](https://doi.org/10.5555/1018408.1018660) as in the **compute_curvatures** tutorial.

**ATP synthase.** Paticle coordinates [ChlamyAnnotations](https://github.com/Chromatin-Structure-Rhythms-Lab/ChlamyAnnotations), associated with ([Kelley et al., 2025, *Mol. Cell*](https://doi.org/10.1016/j.molcel.2025.11.029)).

### Representations

We use **`PleomorphicSurface`** as the notebook-facing API and wrapper that delegates the applied methods to the right underlying geometric representation of the data:

- The underlying representation of IMM is a mesh (`is_mesh=True`) with curvature fields on vertices/triangles.
- ATP synthases can be represented as rays for particle→mesh queries or wrapped as an **`OrientedPointCloud`** for mesh→particle queries.

### Two complementary query directions

| Direction | Query | Seeds on IMM | Question |
|---|---|---|---|
| **Particles → mesh** | Rays from each particle toward the membrane | Hit **triangles** | Which membrane triangle does this particle meet along its casting direction? |
| **Mesh → particles** | Rays from mesh **vertices** toward the particle cloud | Hit **vertices**; regions expanded to **triangles** | Which membrane vertices are close to any particle along the surface normal? |

Use the same distance cutoffs and `surface_radii` in both directions when you want comparable inner/outer bands.

### Workflow

1. Load the IMM mesh (with curvatures) and the ATP motive list; place particle coordinates in the same units as the mesh (nm).
2. **Particles → mesh:** cast rays → get `intersection_data` → extract and save mesh patches.
3. **Mesh → particles:** build a particle point cloud → compute `distance_to_pointcloud` → get `intersection_data` → extract and save mesh patches.
4. Compare results and export intersected subregions as meshes for visualization.

## Load dependencies

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from cryocat.utils import geom
from cryocat.core import cryomotl
from cryocat.core import surface
from cryocat.analysis import structure

## Load the data

### IMM mesh

The IMM surface is loaded with **`PleomorphicSurface.read`** and `method="mesh_curvatures"`. That reads a VTP triangle mesh and attaches principal, mean, and Gaussian curvature fields produced in the **compute_curvatures** notebook. Set the units at load time so distances and curvature units stay consistent.

After loading, `IMM_surface.is_mesh` should be `True`.

In [3]:
# General paths
input_path = "inputs/"
output_path = "outputs/"

In [4]:
IMM_mesh_path = input_path + "IMM_mesh_screenedPoisson_curvatures.vtp"

In [5]:
# Read in mesh with curvatures and directly wrap as a PleomorphicSurface instance
IMM = structure.PleomorphicSurface.read(
    IMM_mesh_path,
    method="mesh_curvatures",
    units="nm"
)

Loading triangle mesh with curvatures: inputs/IMM_mesh_screenedPoisson_curvatures.vtp
Loaded: 133654 vertices, 260614 faces
Coordinate units: nm
Curvatures will be in: nm^(-1) (principal/mean), nm^(-2) (Gaussian)
Curvature data loaded from file
  Principal curvatures: k1=[-4.830991e-01, 5.054894e-01], k2=[-8.593560e-01, 1.324550e-01]
  Mean curvature: [-5.168882e-01, 1.803246e-01]
  Gaussian curvature: [-8.929300e-02, 2.630900e-01]


In [6]:
print(f"IMM.is_mesh: {IMM.is_mesh}")

IMM.is_mesh: True


### ATP synthase motive list

Particles coordinates are rescaled by the pixel size to match the mesh coordinate system. Adjust this factor for your data.

Particle normals are computed as the z-axis in the motive list frame.

In [7]:
atp_motl_path = input_path + "atp_bin4_trimmed.em"

In [8]:
# Load particle motive list
atp_synthase_motl = cryomotl.Motl.load(atp_motl_path)

In [9]:
# Get particle coordinates and re-scale with respect to the mesh (this depends on your data)

pixel_size = 0.196 # in nm to match the pixel size of the meshes
atp_synthase_xyz = atp_synthase_motl.get_coordinates() * pixel_size

In [10]:
# Get z-axis normals from the motive list

atp_synthase_rotations = atp_synthase_motl.get_rotations()
atp_synthase_normals = atp_synthase_rotations.apply([0, 0, 1])

## Ray casting: particles → mesh

Rays start at each ATP synthase position and shoot in the **−z** normal direction toward the IMM mesh. When `one_hit_per_target=True`, the triangle with the shortest ray source-mesh distance is stored.

**Ray direction:** In this dataset, the particle orientations were first inspected in ChimeraX. Casting along **−z** matches how they are oriented relative to the membrane. Verify for your data.

| Step | Method | Purpose |
|---|---|---|
| Build rays | `geom.construct_rays` | `(N, 6)` array: origin + direction per particle |
| Intersect | `IMM.ray_intersections` | Open3D ray cast against the mesh |
| Analyze | `IMM.intersection_data` | Hit table, radial regions, curvature summary |
| Extract | `IMM.extract_region` | Sub-meshes for visualization / export |

- IMM mesh colored by Gaussian curvature
- Spheres are ATP synthases as per the original motive list
<img src="media/IMM_fullmesh_Gaussiancurvature+ATPsynthases.png" width="800"/>

In [11]:
# Construct rays from particles in minus z-direction

atp_synthase_rays = geom.construct_rays(
    points=atp_synthase_xyz,
    normals=atp_synthase_normals,
    reverse_direction=True 
)

In [12]:
# Compute intersections from the particles to the mesh in the direction of the rays

cast_rays_on_IMM_mesh = IMM.ray_intersections(
    rays=atp_synthase_rays,
    one_hit_per_target=True, # stores the triangle with the shortest distance from the point source
)

### Get intersected triangles and grow membrane regions

`intersection_data` turns raw ray triangle hits into a summary DataFrame and and optionally gets informatio for certain regions of the mesh.

| Parameter | Value here | Meaning |
|---|---:|---|
| `query_type` | `"ray"` | Parse output of `ray_intersections` |
| `max_distance_source_target` | `20` | Drop hits farther than 20 nm (empirical; set after visual inspection) |
| `source_id_name` | `"particle_id"` | Ray / particle index |
| `target_id_name` | `"triangle_id"` | Intersected triangle on the mesh |
| `surface_radii` | `[9.0, 18.0]` | Grow regions on the mesh around each intersected triangle: inner disk ≤ 9 nm; annulus 9–18 nm |
| `include_curvatures` | `True` | Attach mean / Gaussian curvature at each intersected triangle; summarize per region |

**Why the radii?** ~9 nm approximates the radius of ATP synthase base from STA. 18 nm defines an outer band for “surrounding” membrane. Regions are grown in the mesh metric from seed intersecting triangles.

Inspect **`region_summary`**: compare `hit triangles`, `r <= 9 nm`, `9 < r <= 18 nm`, etc. This is the main curvature comparison table for the particles→mesh path.

In [13]:
# Wrapper to get intersection data

particle_to_mesh_data = IMM.intersection_data(
    result=cast_rays_on_IMM_mesh,
    query_type="ray",
    max_distance_source_target=20, # nm, same as mesh
    source_id_name="particle_id",
    target_id_name="triangle_id",
    surface_radii=[9.0, 18.0], # nm, same as mesh; what area sizes to grow on the mesh
    include_curvatures=True,
)

particle_to_mesh_hits = particle_to_mesh_data["hits"] # get intersected triangles
particle_to_mesh_data["region_summary"] # get summary of regions on the mesh, chosen with surface_radii

,region,n_triangles,mean_curvature_mean,mean_curvature_median,gaussian_curvature_mean,gaussian_curvature_median
0,hit triangles,360,-0.019621,-0.030229,0.000497,0.000641
1,r <= 9 nm,71947,-0.020891,-0.030642,0.000576,0.000674
2,r <= 18 nm,107335,-0.020483,-0.026545,0.000249,0.000397
3,9 < r <= 18 nm,35388,-0.019654,-0.019108,-0.000416,-0.000062


### Extract and save membrane patches

`extract_region` takes a triangle index array from the summary DaatFrame above and returns a new **`PleomorphicSurface`** sub-mesh for a respective region of interest (curvature fields preserved).

| Region key | Output file (example) | Content |
|---|---|---|
| `"r <= 9 nm"` (or different) | `.vtp` mesh with curvatures | Membrane directly under particles |
| `"9 < r <= 18 nm"` (or different) | `.vtp` mesh with curvatures  | Surrounding annulus |

In [ ]:
# Grow regions starting from the intersected triangle
# Region within an ATP synthase particle

IMM_inner_patch = IMM.extract_region(
    particle_to_mesh_data["regions"]["r <= 9 nm"],
    element="triangles",
)

# Surrounding annulus
IMM_outer_patch = IMM.extract_region(
    particle_to_mesh_data["regions"]["9 < r <= 18 nm"],
    element="triangles",
)

In [15]:
# Save the extracted meshes

IMM_inner_patch.save(
    output_path = output_path + "IMM_below9nm_patch_particlestomesh.vtp",
    format="vtp",
    include_curvatures=True,
)

IMM_outer_patch.save(
    output_path = output_path + "IMM_9nmto18nm_patch_particlestomesh.vtp",
    format="vtp",
    include_curvatures=True,
)

Saved mesh with curvatures to outputs/IMM_below9nm_patch_particlestomesh.vtp
  Format: VTP (VTK PolyData)
  Vertices: 39,387
  Faces: 71,947
  Scalar fields (5):
    - mean_curvature: [-4.036615e-01, 1.119790e-01]
    - gaussian_curvature: [-3.217454e-02, 9.952110e-02]
    - k1: [-1.666755e-01, 1.986733e-01]
    - k2: [-7.482412e-01, 5.034715e-02]
    - curvature_anisotropy: [2.241403e-04, 7.015738e-01]
  Vector fields (3): normals, principal_direction_1, principal_direction_2
  Curvature units: nm^(-1) (principal/mean), nm^(-2) (Gaussian)
Saved mesh with curvatures to outputs/IMM_9nmto18nm_patch_particlestomesh.vtp
  Format: VTP (VTK PolyData)
  Vertices: 22,224
  Faces: 35,388
  Scalar fields (5):
    - mean_curvature: [-4.362034e-01, 1.197936e-01]
    - gaussian_curvature: [-4.719297e-02, 1.177193e-01]
    - k1: [-1.668447e-01, 2.111054e-01]
    - k2: [-7.195559e-01, 6.802757e-02]
    - curvature_anisotropy: [3.295864e-05, 7.606541e-01]
  Vector fields (3): normals, principal_direct

{'format': 'VTP',
 'path': 'outputs/IMM_9nmto18nm_patch_particlestomesh.vtp',
 'n_vertices': 22224,
 'n_faces': 35388,
 'coordinate_units': 'nm',
 'scalar_fields': ['mean_curvature',
  'gaussian_curvature',
  'k1',
  'k2',
  'curvature_anisotropy'],
 'vector_fields': ['normals',
  'principal_direction_1',
  'principal_direction_2'],
 'statistics': {'mean_curvature': {'min': -0.43620338744383036,
   'max': 0.11979361095028015,
   'mean': -0.020370677445557574},
  'gaussian_curvature': {'min': -0.04719296892544032,
   'max': 0.11771930286401895,
   'mean': -0.0003841990467344351},
  'k1': {'min': -0.16684471883735424,
   'max': 0.21110538547552327,
   'mean': 0.01206824217007824},
  'k2': {'min': -0.7195558784164204,
   'max': 0.06802756509271167,
   'mean': -0.052809597061193383},
  'curvature_anisotropy': {'min': 3.29586417993355e-05,
   'max': 0.7606540562329733,
   'mean': 0.06487783923127162}}}

- r <= 9 nm
<img src="media/IMM_below9nm_patch_particlestomesh_Gaussiancurvature.png" width="800"/>
- 9 < r <= 18 nm
<img src="media/IMM_9nmto18nm_patch_particlestomesh_Gaussiancurvature.png" width="800"/>	

## Mesh → particles

The reverse path asks: **which mesh vertices are close to the ATP synthase cloud** when casting from the mesh toward particles? This approach can be useful if information on the orientation of the particles is missing.

1. Build an **`OrientedPointCloud`** from particle coordinates (normals optional).
2. Wrap it in **`PleomorphicSurface`**.
3. Call **`distance_to_pointcloud`** on the mesh with the particle cloud as **target**.

For a screened-Poisson mesh, vertex normals are not always consistently oriented. **`bidirectional=True`** casts along ±normal and keeps the closer hit. **`one_hit_per_target=True`** keeps one mesh vertex per particle (closest approach), analogous to one triangle per particle in the ray path.

| Parameter | Value here | Meaning |
|---|---:|---|
| `max_distance` | `20.0` | Same distance cutoff as the ray path (nm) |
| `bidirectional` | `True` | Try both normal directions |
| `one_hit_per_target` | `True` | One source vertex per target particle |

In [16]:
# Construct an oriented point cloud from the ATP synthase positions
atp_synthase_pc = surface.OrientedPointCloud()

# Write vertices
atp_synthase_pc.vertices = atp_synthase_xyz

# Write normals (optional, the code runs just the same for unoriented data in this case)
atp_synthase_pc.normals = atp_synthase_normals  # optional but useful

# Convert to PleomorphicSurface instance
atp_synthase_surface = structure.PleomorphicSurface(atp_synthase_pc)

In [17]:
# Intersect the ATP synthase point cloud starting from the mesh
# All mesh vertices are used
# If one_hit_per_target, return only the vertex that has the closest distance to the intersected points

intersect_mesh_with_pc = IMM.distance_to_pointcloud(
    target=atp_synthase_surface,
    max_distance=20.0,
    bidirectional=True,
    one_hit_per_target=True
)

Ray casting: 388/133654 mesh vertices intersect with point cloud


### Get intersecting vertices and grow membrane regions from mesh→particle hits

Same helper as above, with **`query_type="distance_to_pointcloud"`**.

| Parameter | Value here | Meaning |
|---|---:|---|
| `source_id_name` | `"vertex_id"` | Mesh vertex that intersects the cloud |
| `target_id_name` | `"particle_id"` | Closest particle index |
| `surface_radii` | `[9.0, 18.0]` | Same inner / outer radii as particles→mesh |
| `surface_element` | default (`vertices` for this query) | Seeds are intersect vertices; regions expand to touching triangles for curvature summaries |

Compare summarizing DataFrame to the particles→mesh table. Counts and curvature means might differ slightly (different seeding: triangles vs vertices), but trends in inner vs outer bands should be comparable if set correctly.

In [18]:
# Compute intersection data
# Regions are grown from the intersecting vertices (important to be one hit per target)

mesh_to_particles_data = IMM.intersection_data(
    result=intersect_mesh_with_pc,
    query_type="distance_to_pointcloud",
    max_distance_source_target=20.0,
    source_id_name="vertex_id",
    target_id_name="particle_id",
    surface_radii=[9.0, 18.0],
    surface_element="vertices",  # optional: default for mesh distance queries
    include_curvatures=True,
)

mesh_to_particle_hits = mesh_to_particles_data["hits"]
mesh_to_particles_data["region_summary"]

,region,n_triangles,mean_curvature_mean,mean_curvature_median,gaussian_curvature_mean,gaussian_curvature_median
0,hit triangles,2268,-0.020628,-0.033160,0.000754,0.000720
1,r <= 9 nm,75111,-0.020805,-0.030550,0.000504,0.000634
2,r <= 18 nm,112728,-0.019338,-0.025354,0.000217,0.000353
3,9 < r <= 18 nm,37617,-0.016409,-0.015810,-0.000356,-0.000003


### Extract and save membrane patches (mesh → particles)

Extract annular patches the same way as in the particles→mesh section, but index regions from the new DataFrame.

In [19]:
# Extract sub-mesh region
# Region within an ATP synthase particle

IMM_inner_patch_meshtopc = IMM.extract_region(
    mesh_to_particles_data["regions"]["r <= 9 nm"],
    element="triangles",
)

# Surrounding annulus
IMM_outer_patch_meshtopc = IMM.extract_region(
    mesh_to_particles_data["regions"]["9 < r <= 18 nm"],
    element="triangles",
)

In [20]:
# Save the extracted meshes

IMM_inner_patch_meshtopc.save(
    output_path = output_path + "IMM_below9nm_patch_meshtopc.vtp",
    format="vtp",
    include_curvatures=True,
)

IMM_outer_patch_meshtopc.save(
    output_path = output_path + "IMM_9nmto18nm_patch_meshtopc.vtp",
    format="vtp",
    include_curvatures=True,
)

Saved mesh with curvatures to outputs/IMM_below9nm_patch_meshtopc.vtp
  Format: VTP (VTK PolyData)
  Vertices: 41,204
  Faces: 75,111
  Scalar fields (5):
    - mean_curvature: [-4.362034e-01, 9.801390e-02]
    - gaussian_curvature: [-3.912718e-02, 1.177193e-01]
    - k1: [-1.668447e-01, 1.892228e-01]
    - k2: [-7.482412e-01, 5.034715e-02]
    - curvature_anisotropy: [1.156415e-04, 7.015738e-01]
  Vector fields (3): normals, principal_direction_1, principal_direction_2
  Curvature units: nm^(-1) (principal/mean), nm^(-2) (Gaussian)
Saved mesh with curvatures to outputs/IMM_9nmto18nm_patch_meshtopc.vtp
  Format: VTP (VTK PolyData)
  Vertices: 23,517
  Faces: 37,617
  Scalar fields (5):
    - mean_curvature: [-3.980009e-01, 1.197936e-01]
    - gaussian_curvature: [-4.719297e-02, 9.442748e-02]
    - k1: [-1.762917e-01, 2.851824e-01]
    - k2: [-7.760181e-01, 6.802757e-02]
    - curvature_anisotropy: [1.156415e-04, 7.606622e-01]
  Vector fields (3): normals, principal_direction_1, princip

{'format': 'VTP',
 'path': 'outputs/IMM_9nmto18nm_patch_meshtopc.vtp',
 'n_vertices': 23517,
 'n_faces': 37617,
 'coordinate_units': 'nm',
 'scalar_fields': ['mean_curvature',
  'gaussian_curvature',
  'k1',
  'k2',
  'curvature_anisotropy'],
 'vector_fields': ['normals',
  'principal_direction_1',
  'principal_direction_2'],
 'statistics': {'mean_curvature': {'min': -0.39800089762767943,
   'max': 0.11979361095028015,
   'mean': -0.016946004324438463},
  'gaussian_curvature': {'min': -0.04719296892544032,
   'max': 0.09442748006071483,
   'mean': -0.0003458651971554649},
  'k1': {'min': -0.17629172841642776,
   'max': 0.2851824379952576,
   'mean': 0.012870640489422349},
  'k2': {'min': -0.7760180805330399,
   'max': 0.06802756509271167,
   'mean': -0.04676264913829927},
  'curvature_anisotropy': {'min': 0.00011564154540416594,
   'max': 0.7606621993125067,
   'mean': 0.059633289627721624}}}